# LUPW Flow Traffic Light System — Training Notebook

## Project Overview

This notebook trains a **multi-output CNN** (MobileNetV2 backbone) to read **rotameter flow meters** from images captured by a Raspberry Pi Camera Module 2.

**The model has two outputs:**
1. **Flow regression** — predicts the actual flow value in GPM (continuous float)
2. **Zero-flow classifier** — binary classification: is the flow zero? (yes/no)

**Deployment target:** Raspberry Pi 4 with TFLite, controlling GPIO traffic lights:
- **Red LED** → no flow (0 GPM)
- **Amber LED** → rinse mode (≤10 GPM)
- **Green LED** → online (>10 GPM)

**Why MobileNetV2?** It's lightweight enough to run inference on the Pi 4 in real-time while still being accurate with transfer learning from ImageNet.

**Why dual outputs?** A regression model alone might predict 0.02 instead of exactly 0.0 — the dedicated binary classifier head ensures robust zero-flow detection for reliable traffic light switching.

## Section 1: Install Dependencies and Import Libraries

Install all required packages. Run this on your **training machine** (PC/workstation with GPU recommended), NOT on the Raspberry Pi.

In [ ]:
!pip install tensorflow opencv-python matplotlib numpy pandas scikit-learn Pillow

In [ ]:
import os
import glob
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

# Check GPU availability
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
if tf.config.list_physical_devices('GPU'):
    for gpu in tf.config.list_physical_devices('GPU'):
        print(f"  GPU: {gpu.name}")
else:
    print("  Training will use CPU (slower). GPU recommended for faster training.")

## Section 2: Dataset Structure and Preparation Guide

### How to Collect Your Training Data

1. **Mount your Pi Camera Module 2** in front of the rotameter at a fixed angle (the same position it will be deployed in)
2. **Photograph the rotameter** at different flow levels. Recommended:
   - **Flow levels:** 0, 0.5, 1.0, 1.5, 2.0, ... up to your rotameter's max scale
   - **Images per level:** 50–100 (vary lighting slightly, minor camera angle changes)
   - **Total:** Aim for 500–2000 images across all flow levels
3. **Organize into folders** where each folder name is the flow value:

```
dataset/
├── 0.0/          ← images at zero flow
│   ├── img_001.jpg
│   ├── img_002.jpg
│   └── ...
├── 0.5/          ← images at 0.5 flow
│   ├── img_001.jpg
│   └── ...
├── 1.0/
├── 1.5/
├── ...
└── 10.0/         ← images at max flow (adjust to your scale)
```

4. The code below will automatically:
   - Scan the folders and build a `labels.csv` mapping each image to its flow value
   - Create a binary `is_zero_flow` column
   - Split into train/val/test sets

### Tips for Good Data
- Keep the camera distance consistent (~15–30 cm)
- Include slight lighting variations (overhead light, side light, dimmer)
- Ensure the float/ball is clearly visible in every image
- Don't include images where the rotameter is obscured or out of frame

In [ ]:
# ── Configuration ──────────────────────────────────────────────────
DATASET_DIR = "dataset"       # Root folder containing flow-level subfolders
IMAGE_SIZE = (224, 224)       # MobileNetV2 input size
LABELS_CSV = "dataset/labels.csv"

# ── Generate labels.csv from folder structure ─────────────────────
def generate_labels_csv(dataset_dir, output_csv):
    """
    Scan dataset_dir for subfolders named by flow value (e.g., '0.0', '1.5', '10.0').
    Each subfolder contains .jpg/.jpeg/.png images at that flow level.
    Produces a CSV: image_path, flow_value, is_zero_flow
    """
    records = []
    if not os.path.exists(dataset_dir):
        print(f"ERROR: Dataset directory '{dataset_dir}' not found!")
        print(f"Please create the folder structure as described above.")
        return None

    for folder_name in sorted(os.listdir(dataset_dir)):
        folder_path = os.path.join(dataset_dir, folder_name)
        if not os.path.isdir(folder_path):
            continue
        try:
            flow_value = float(folder_name)
        except ValueError:
            continue  # Skip non-numeric folders

        for img_file in sorted(os.listdir(folder_path)):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                img_path = os.path.join(folder_path, img_file)
                is_zero = 1 if flow_value == 0.0 else 0
                records.append({
                    "image_path": img_path,
                    "flow_value": flow_value,
                    "is_zero_flow": is_zero
                })

    if len(records) == 0:
        print("WARNING: No images found! Check your dataset folder structure.")
        return None

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"Generated {output_csv} with {len(df)} images")
    print(f"  Flow levels found: {sorted(df['flow_value'].unique())}")
    print(f"  Zero-flow images: {df['is_zero_flow'].sum()}")
    print(f"  Non-zero images:  {(~df['is_zero_flow'].astype(bool)).sum()}")
    return df

# Generate the CSV
df = generate_labels_csv(DATASET_DIR, LABELS_CSV)

In [ ]:
# ── Verify Dataset Integrity ───────────────────────────────────────
def verify_dataset(df):
    """Check for missing files and show distribution stats."""
    if df is None:
        print("No dataset to verify. Please populate the dataset/ folder first.")
        return

    # Check for missing images
    missing = []
    for _, row in df.iterrows():
        if not os.path.exists(row["image_path"]):
            missing.append(row["image_path"])

    if missing:
        print(f"WARNING: {len(missing)} image files not found!")
        for m in missing[:5]:
            print(f"  Missing: {m}")
    else:
        print(f"All {len(df)} image files verified.")

    # Check image readability on a sample
    sample = df.sample(min(10, len(df)))
    bad = 0
    for _, row in sample.iterrows():
        img = cv2.imread(row["image_path"])
        if img is None:
            bad += 1
            print(f"  Cannot read: {row['image_path']}")
    if bad == 0:
        print(f"Sample of {len(sample)} images all readable.")

    # Distribution
    print(f"\nDataset Summary:")
    print(f"  Total images: {len(df)}")
    print(f"  Flow range: {df['flow_value'].min():.1f} – {df['flow_value'].max():.1f}")
    print(f"  Mean flow:  {df['flow_value'].mean():.2f}")
    print(f"  Std flow:   {df['flow_value'].std():.2f}")

if df is not None:
    verify_dataset(df)

## Section 3: Load and Explore the Dataset

Load the labels CSV, display summary statistics, visualize sample images at different flow levels, and plot the label distribution. Then split into train (70%), validation (15%), and test (15%) sets with stratified binning.

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────
df = pd.read_csv(LABELS_CSV)
print(df.describe())
print(f"\nFlow value distribution:")
print(df["flow_value"].value_counts().sort_index())

# ── Visualize sample images at different flow levels ──────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Sample Images at Different Flow Levels", fontsize=14)

unique_flows = sorted(df["flow_value"].unique())
sample_flows = np.linspace(0, len(unique_flows) - 1, 8, dtype=int)
sample_flows = [unique_flows[i] for i in sample_flows]

for ax, flow_val in zip(axes.flat, sample_flows):
    subset = df[df["flow_value"] == flow_val]
    if len(subset) > 0:
        row = subset.sample(1).iloc[0]
        img = cv2.imread(row["image_path"])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        ax.set_title(f"Flow: {flow_val:.1f}", fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

# ── Label distribution histogram ─────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(df["flow_value"], bins=30, edgecolor="black", alpha=0.7, color="steelblue")
ax1.set_xlabel("Flow Value")
ax1.set_ylabel("Count")
ax1.set_title("Flow Value Distribution")

zero_counts = df["is_zero_flow"].value_counts()
ax2.bar(["Non-Zero Flow", "Zero Flow"], [zero_counts.get(0, 0), zero_counts.get(1, 0)],
        color=["green", "red"], edgecolor="black")
ax2.set_ylabel("Count")
ax2.set_title("Zero vs Non-Zero Flow")
plt.tight_layout()
plt.show()

In [ ]:
# ── Train / Validation / Test Split (70/15/15) with stratified binning ──
# Bin flow values for stratified splitting (ensures each split has similar distribution)
n_bins = min(10, len(df["flow_value"].unique()))
df["flow_bin"] = pd.cut(df["flow_value"], bins=n_bins, labels=False)

# First split: 70% train, 30% temp
df_train, df_temp = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["flow_bin"]
)

# Second split: 50/50 of temp → 15% val, 15% test
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, random_state=42, stratify=df_temp["flow_bin"]
)

# Drop the binning column
df_train = df_train.drop(columns=["flow_bin"]).reset_index(drop=True)
df_val = df_val.drop(columns=["flow_bin"]).reset_index(drop=True)
df_test = df_test.drop(columns=["flow_bin"]).reset_index(drop=True)

print(f"Train: {len(df_train)} images")
print(f"Val:   {len(df_val)} images")
print(f"Test:  {len(df_test)} images")
print(f"\nTrain flow range: {df_train['flow_value'].min():.1f} – {df_train['flow_value'].max():.1f}")
print(f"Val   flow range: {df_val['flow_value'].min():.1f} – {df_val['flow_value'].max():.1f}")
print(f"Test  flow range: {df_test['flow_value'].min():.1f} – {df_test['flow_value'].max():.1f}")

## Section 4: Data Preprocessing and Augmentation

- Resize all images to **224×224** (MobileNetV2 input size)
- Normalize pixel values to **[0, 1]**
- Apply data augmentation: random brightness, contrast, slight rotation (±5°), and minor translation to simulate camera wobble
- Build efficient `tf.data.Dataset` pipelines with prefetching

In [ ]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# ── Normalize flow values to [0, 1] range for better training ─────
FLOW_MAX = df["flow_value"].max()
print(f"Max flow value in dataset: {FLOW_MAX}")
print(f"Flow values will be normalized to [0, 1] during training and de-normalized during inference.")

def load_and_preprocess(image_path, flow_value, is_zero):
    """Load image, resize to 224x224, normalize to [0,1]."""
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    # Normalize flow to [0, 1]
    flow_normalized = flow_value / FLOW_MAX
    return img, (flow_normalized, is_zero)

def augment(img, labels):
    """Apply data augmentation for training images."""
    # Random brightness
    img = tf.image.random_brightness(img, max_delta=0.15)
    # Random contrast
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    # Random slight rotation (±5 degrees) via affine transform
    angle = tf.random.uniform([], -5.0, 5.0) * (np.pi / 180.0)
    img = tf.keras.layers.RandomRotation(factor=(-5/360, 5/360))(tf.expand_dims(img, 0))[0]
    # Random translation (±5% shift)
    img = tf.keras.layers.RandomTranslation(
        height_factor=0.05, width_factor=0.05
    )(tf.expand_dims(img, 0))[0]
    # Clip to valid range
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, labels

def build_dataset(dataframe, augment_data=False, shuffle=False):
    """Build a tf.data.Dataset from a dataframe."""
    paths = dataframe["image_path"].values
    flows = dataframe["flow_value"].values.astype(np.float32)
    zeros = dataframe["is_zero_flow"].values.astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, flows, zeros))
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe))
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

# Build datasets
train_ds = build_dataset(df_train, augment_data=True, shuffle=True)
val_ds = build_dataset(df_val)
test_ds = build_dataset(df_test)

# Verify a batch
for images, (flow_labels, zero_labels) in train_ds.take(1):
    print(f"Image batch shape:  {images.shape}")
    print(f"Flow labels shape:  {flow_labels.shape}")
    print(f"Zero labels shape:  {zero_labels.shape}")
    print(f"Flow range in batch: {flow_labels.numpy().min():.3f} – {flow_labels.numpy().max():.3f}")
    print(f"Zero labels in batch: {zero_labels.numpy()[:10]}")

## Section 5: Define the Multi-Output CNN Model

**Architecture: MobileNetV2 + Dual Heads**

- **Backbone:** MobileNetV2 pretrained on ImageNet (frozen initially for transfer learning)
- **Head 1 — Flow Regression:** GlobalAveragePooling → Dense(128) → Dropout → Dense(1, linear) → predicts normalized flow value
- **Head 2 — Zero-Flow Classifier:** GlobalAveragePooling → Dense(64) → Dropout → Dense(1, sigmoid) → predicts probability of zero flow

**Why two heads?**
A regression model might predict 0.02 instead of exactly 0.0. The dedicated binary classifier ensures we never miss a zero-flow event — critical for the traffic light safety signal.

In [ ]:
# ── Build the multi-output model ──────────────────────────────────
def build_model():
    # MobileNetV2 backbone (frozen for phase 1)
    backbone = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    backbone.trainable = False  # Freeze for initial training

    inputs = keras.Input(shape=(224, 224, 3), name="input_image")
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)

    # ── Head 1: Flow Regression ──
    flow_branch = layers.Dense(128, activation="relu", name="flow_dense1")(x)
    flow_branch = layers.Dropout(0.3, name="flow_dropout")(flow_branch)
    flow_branch = layers.Dense(64, activation="relu", name="flow_dense2")(flow_branch)
    flow_output = layers.Dense(1, activation="linear", name="flow_output")(flow_branch)

    # ── Head 2: Zero-Flow Binary Classifier ──
    zero_branch = layers.Dense(64, activation="relu", name="zero_dense1")(x)
    zero_branch = layers.Dropout(0.3, name="zero_dropout")(zero_branch)
    zero_output = layers.Dense(1, activation="sigmoid", name="zero_output")(zero_branch)

    model = Model(inputs=inputs, outputs=[flow_output, zero_output])
    return model, backbone

model, backbone = build_model()

# ── Custom zero-flow accuracy metric ─────────────────────────────
class ZeroFlowAccuracy(keras.metrics.Metric):
    """Tracks binary accuracy of the zero-flow detection head."""
    def __init__(self, threshold=0.5, **kwargs):
        super().__init__(name="zero_flow_acc", **kwargs)
        self.threshold = threshold
        self.correct = self.add_weight(name="correct", initializer="zeros")
        self.total = self.add_weight(name="total", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        pred_labels = tf.cast(y_pred >= self.threshold, tf.float32)
        matches = tf.cast(tf.equal(y_true, pred_labels), tf.float32)
        self.correct.assign_add(tf.reduce_sum(matches))
        self.total.assign_add(tf.cast(tf.size(y_true), tf.float32))

    def result(self):
        return self.correct / (self.total + keras.backend.epsilon())

    def reset_state(self):
        self.correct.assign(0.0)
        self.total.assign(0.0)

# ── Compile with combined loss ────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "flow_output": "mse",
        "zero_output": "binary_crossentropy",
    },
    loss_weights={
        "flow_output": 1.0,     # Regression loss weight
        "zero_output": 0.5,     # Classification loss weight
    },
    metrics={
        "flow_output": ["mae"],
        "zero_output": [ZeroFlowAccuracy()],
    },
)

model.summary()

## Section 6: Traffic Light Decision Logic

The traffic light has **three states** based on flow rate:

| State | Condition | LED |
|---|---|---|
| **RED** | No flow (0 GPM) | Red ON |
| **AMBER** | Rinse mode (>0 and ≤10 GPM) | Amber ON |
| **GREEN** | Online (>10 GPM) | Green ON |

**Zero-flow detection** uses both model outputs for reliability:
1. **Regression head** predicts flow value → if near zero → *likely no flow*
2. **Classifier head** predicts zero probability → if > 0.5 → *likely no flow*
3. **Final decision:** Flow is declared **zero** when both heads agree, OR classifier is very confident (>0.8)

Once zero-flow is determined, the remaining flow is categorised by the 10 GPM rinse threshold.

In [ ]:
# ── Traffic light decision function ───────────────────────────────
FLOW_THRESHOLD = 0.05  # 5% of max scale = "near zero" for regression head
RINSE_GPM = 10.0       # At or below this GPM = rinse (amber)

def is_zero_flow(predicted_flow_normalized, zero_probability):
    """
    Determine if flow is zero using both model heads.

    Args:
        predicted_flow_normalized: float in [0, 1] from regression head
        zero_probability: float in [0, 1] from classifier head

    Returns:
        bool: True if flow is considered zero
    """
    # Both heads agree: regression says near-zero AND classifier says zero
    if predicted_flow_normalized < FLOW_THRESHOLD and zero_probability > 0.5:
        return True
    # Classifier is very confident on its own
    if zero_probability > 0.8:
        return True
    return False

def get_traffic_state(flow_gpm, zero_probability=None, flow_normalized=None):
    """
    Determine traffic light state from flow reading.

    Args:
        flow_gpm: Actual flow in GPM
        zero_probability: Optional classifier output for robust zero detection
        flow_normalized: Optional normalized flow for zero detection

    Returns:
        str: 'RED', 'AMBER', or 'GREEN'
    """
    # Check for zero flow using dual-head logic if available
    if flow_normalized is not None and zero_probability is not None:
        if is_zero_flow(flow_normalized, zero_probability):
            return "RED"
    elif flow_gpm <= 0:
        return "RED"

    # Flow is non-zero: check rinse threshold
    if flow_gpm <= RINSE_GPM:
        return "AMBER"
    else:
        return "GREEN"

# Demo the logic
test_cases = [
    (0.00, 0.95, "Zero flow - both heads agree"),
    (0.02, 0.70, "Near-zero + classifier agrees → RED"),
    (0.08, 0.10, "Low flow (~8% of max) → AMBER (rinse)"),
    (0.15, 0.05, "~15% of max, above 10 GPM if max=100 → GREEN"),
    (0.50, 0.10, "Mid flow - clearly online → GREEN"),
    (0.03, 0.30, "Near-zero but classifier unsure → AMBER (not zero)"),
    (0.10, 0.85, "Low flow but classifier very confident zero → RED"),
]

# Assume FLOW_MAX for demo
FLOW_MAX_DEMO = 100.0
print("Traffic Light Decision Logic Test Cases (assuming max_flow=100 GPM):")
print("-" * 75)
for flow_norm, zero_prob, desc in test_cases:
    flow_gpm = flow_norm * FLOW_MAX_DEMO
    state = get_traffic_state(flow_gpm, zero_prob, flow_norm)
    icons = {"RED": "🔴 RED   ", "AMBER": "🟡 AMBER ", "GREEN": "🟢 GREEN "}
    print(f"  flow={flow_gpm:5.1f} GPM, zero_prob={zero_prob:.2f} → {icons[state]} ({desc})")

## Section 7: Train the Model

**Two-phase training strategy:**
1. **Phase 1 (20 epochs):** Frozen MobileNetV2 backbone, only train the custom heads. Learning rate: 1e-3.
2. **Phase 2 (30 epochs):** Unfreeze the top 30 layers of MobileNetV2 and fine-tune everything. Learning rate: 1e-5.

This prevents the pretrained weights from being destroyed by large gradients early in training.

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

callbacks = [
    ModelCheckpoint(
        "checkpoints/best_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1,
    ),
    TensorBoard(log_dir="logs"),
]

# ══════════════════════════════════════════════════════════════════
# PHASE 1: Train with frozen backbone (warm up custom heads)
# ══════════════════════════════════════════════════════════════════
print("=" * 60)
print("PHASE 1: Training custom heads (backbone frozen)")
print("=" * 60)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
)

# ══════════════════════════════════════════════════════════════════
# PHASE 2: Fine-tune top layers of MobileNetV2
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PHASE 2: Fine-tuning (unfreezing top 30 backbone layers)")
print("=" * 60)

# Unfreeze the top 30 layers of MobileNetV2
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss={
        "flow_output": "mse",
        "zero_output": "binary_crossentropy",
    },
    loss_weights={
        "flow_output": 1.0,
        "zero_output": 0.5,
    },
    metrics={
        "flow_output": ["mae"],
        "zero_output": [ZeroFlowAccuracy()],
    },
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
)

print("\nTraining complete!")

In [ ]:
# ── Plot training curves ──────────────────────────────────────────
def plot_training_history(h1, h2):
    """Combine phase 1 and phase 2 histories and plot."""
    # Merge histories
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history.get(key, [])

    epochs = range(1, len(combined["loss"]) + 1)
    phase1_end = len(h1.history["loss"])

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Training History", fontsize=14)

    # Total loss
    axes[0, 0].plot(epochs, combined["loss"], "b-", label="Train")
    axes[0, 0].plot(epochs, combined["val_loss"], "r-", label="Val")
    axes[0, 0].axvline(x=phase1_end, color="gray", linestyle="--", label="Fine-tune start")
    axes[0, 0].set_title("Total Loss")
    axes[0, 0].legend()
    axes[0, 0].set_xlabel("Epoch")

    # Flow MAE
    axes[0, 1].plot(epochs, combined["flow_output_mae"], "b-", label="Train")
    axes[0, 1].plot(epochs, combined["val_flow_output_mae"], "r-", label="Val")
    axes[0, 1].axvline(x=phase1_end, color="gray", linestyle="--")
    axes[0, 1].set_title("Flow Regression MAE (normalized)")
    axes[0, 1].legend()
    axes[0, 1].set_xlabel("Epoch")

    # Flow loss
    axes[1, 0].plot(epochs, combined["flow_output_loss"], "b-", label="Train")
    axes[1, 0].plot(epochs, combined["val_flow_output_loss"], "r-", label="Val")
    axes[1, 0].axvline(x=phase1_end, color="gray", linestyle="--")
    axes[1, 0].set_title("Flow Regression Loss (MSE)")
    axes[1, 0].legend()
    axes[1, 0].set_xlabel("Epoch")

    # Zero-flow accuracy
    axes[1, 1].plot(epochs, combined["zero_output_zero_flow_acc"], "b-", label="Train")
    axes[1, 1].plot(epochs, combined["val_zero_output_zero_flow_acc"], "r-", label="Val")
    axes[1, 1].axvline(x=phase1_end, color="gray", linestyle="--")
    axes[1, 1].set_title("Zero-Flow Classification Accuracy")
    axes[1, 1].legend()
    axes[1, 1].set_xlabel("Epoch")

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150)
    plt.show()
    print("Saved training_curves.png")

plot_training_history(history_phase1, history_phase2)

## Section 8: Evaluate Model Performance

Evaluate on the held-out test set:
- **Flow regression:** MAE, RMSE, predicted vs actual scatter plot
- **Zero-flow classification:** Accuracy, Precision, Recall, F1, Confusion Matrix
- **Error analysis:** Show the worst predictions for debugging

In [ ]:
# ── Load best model ───────────────────────────────────────────────
best_model = keras.models.load_model(
    "checkpoints/best_model.keras",
    custom_objects={"ZeroFlowAccuracy": ZeroFlowAccuracy},
)

# ── Predict on test set ──────────────────────────────────────────
all_flow_true = []
all_zero_true = []
all_flow_pred = []
all_zero_pred = []
all_images = []

for images, (flow_labels, zero_labels) in test_ds:
    flow_preds, zero_preds = best_model.predict(images, verbose=0)
    all_flow_true.extend(flow_labels.numpy().flatten())
    all_zero_true.extend(zero_labels.numpy().flatten())
    all_flow_pred.extend(flow_preds.flatten())
    all_zero_pred.extend(zero_preds.flatten())
    all_images.extend(images.numpy())

all_flow_true = np.array(all_flow_true)
all_zero_true = np.array(all_zero_true)
all_flow_pred = np.array(all_flow_pred)
all_zero_pred = np.array(all_zero_pred)

# De-normalize flow values for interpretable metrics
flow_true_actual = all_flow_true * FLOW_MAX
flow_pred_actual = all_flow_pred * FLOW_MAX

# ── Regression metrics ────────────────────────────────────────────
mae = mean_absolute_error(flow_true_actual, flow_pred_actual)
rmse = np.sqrt(mean_squared_error(flow_true_actual, flow_pred_actual))
print("═" * 50)
print("FLOW REGRESSION METRICS")
print("═" * 50)
print(f"  MAE:  {mae:.3f} (flow units)")
print(f"  RMSE: {rmse:.3f} (flow units)")

# ── Classification metrics ────────────────────────────────────────
zero_pred_binary = (all_zero_pred > 0.5).astype(int)
acc = accuracy_score(all_zero_true, zero_pred_binary)
prec = precision_score(all_zero_true, zero_pred_binary, zero_division=0)
rec = recall_score(all_zero_true, zero_pred_binary, zero_division=0)
f1 = f1_score(all_zero_true, zero_pred_binary, zero_division=0)
cm = confusion_matrix(all_zero_true, zero_pred_binary)

print(f"\n{'═' * 50}")
print("ZERO-FLOW CLASSIFICATION METRICS")
print("═" * 50)
print(f"  Accuracy:  {acc:.3f}")
print(f"  Precision: {prec:.3f}")
print(f"  Recall:    {rec:.3f}")
print(f"  F1 Score:  {f1:.3f}")
print(f"\n  Confusion Matrix:")
print(f"              Pred Non-Zero  Pred Zero")
print(f"  True Non-Zero    {cm[0][0]:>5d}       {cm[0][1]:>5d}")
print(f"  True Zero        {cm[1][0]:>5d}       {cm[1][1]:>5d}")

In [ ]:
# ── Visualization: Predicted vs Actual scatter plot ───────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter: predicted vs actual
axes[0].scatter(flow_true_actual, flow_pred_actual, alpha=0.5, s=20, c="steelblue")
max_val = max(flow_true_actual.max(), flow_pred_actual.max())
axes[0].plot([0, max_val], [0, max_val], "r--", linewidth=2, label="Ideal (y=x)")
axes[0].set_xlabel("Actual Flow")
axes[0].set_ylabel("Predicted Flow")
axes[0].set_title(f"Predicted vs Actual Flow\nMAE={mae:.2f}, RMSE={rmse:.2f}")
axes[0].legend()

# Confusion matrix heatmap
im = axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(["Non-Zero", "Zero"])
axes[1].set_yticklabels(["Non-Zero", "Zero"])
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].set_title("Zero-Flow Confusion Matrix")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm[i][j]), ha="center", va="center",
                     fontsize=16, color="white" if cm[i][j] > cm.max()/2 else "black")

# Error distribution
errors = np.abs(flow_true_actual - flow_pred_actual)
axes[2].hist(errors, bins=30, edgecolor="black", alpha=0.7, color="coral")
axes[2].set_xlabel("Absolute Error (flow units)")
axes[2].set_ylabel("Count")
axes[2].set_title("Prediction Error Distribution")
axes[2].axvline(x=mae, color="red", linestyle="--", label=f"MAE={mae:.2f}")
axes[2].legend()

plt.tight_layout()
plt.savefig("evaluation_results.png", dpi=150)
plt.show()

# ── Show worst predictions ────────────────────────────────────────
print("\nTop 10 Worst Predictions (highest error):")
print("-" * 55)
worst_idx = np.argsort(errors)[-10:][::-1]
for i, idx in enumerate(worst_idx):
    print(f"  {i+1}. Actual: {flow_true_actual[idx]:.1f}, "
          f"Predicted: {flow_pred_actual[idx]:.1f}, "
          f"Error: {errors[idx]:.2f}")

# Show worst 4 images
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("4 Worst Predictions", fontsize=13)
for ax, idx in zip(axes, worst_idx[:4]):
    ax.imshow(all_images[idx])
    ax.set_title(f"Actual: {flow_true_actual[idx]:.1f}\n"
                 f"Pred: {flow_pred_actual[idx]:.1f}\n"
                 f"Err: {errors[idx]:.2f}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Section 9: Export Model for Raspberry Pi (TFLite Conversion)

Convert the trained Keras model to **TensorFlow Lite** with **float16 quantization** for efficient inference on the Raspberry Pi 4.

- Float16 quantization cuts model size in half with minimal accuracy loss
- Optional full integer quantization available for even faster inference

In [ ]:
# ── Convert to TFLite with float16 quantization ─────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

# Save TFLite model
tflite_path = "rotameter_model_fp16.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

tflite_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"TFLite model saved: {tflite_path}")
print(f"Model size: {tflite_size_mb:.2f} MB")

# ── Validate TFLite model against Keras model ────────────────────
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"\nTFLite Model Info:")
print(f"  Input shape:  {input_details[0]['shape']}")
print(f"  Input dtype:  {input_details[0]['dtype']}")
print(f"  Output count: {len(output_details)}")
for i, out in enumerate(output_details):
    print(f"  Output {i}: shape={out['shape']}, name={out['name']}")

# Compare outputs on a few test images
print(f"\nValidation: Comparing Keras vs TFLite predictions on 5 test images...")
sample_images = np.array(all_images[:5], dtype=np.float32)

for i in range(5):
    img = np.expand_dims(sample_images[i], axis=0)

    # Keras prediction
    keras_flow, keras_zero = best_model.predict(img, verbose=0)

    # TFLite prediction
    interpreter.set_tensor(input_details[0]["index"], img)
    interpreter.invoke()
    tflite_flow = interpreter.get_tensor(output_details[0]["index"])
    tflite_zero = interpreter.get_tensor(output_details[1]["index"])

    print(f"  Image {i+1}: Keras flow={keras_flow[0][0]:.4f}, "
          f"TFLite flow={tflite_flow[0][0]:.4f}, "
          f"Diff={abs(keras_flow[0][0] - tflite_flow[0][0]):.6f}")

print(f"\nDone! Transfer '{tflite_path}' to your Raspberry Pi.")
print(f"  scp {tflite_path} pi@<PI_IP>:~/lupw-project/")

## Section 10: Raspberry Pi Hardware Setup Guide

See `setup_guide.md` for the full wiring diagram (including 5m cable transistor driver). Quick summary:

| Signal | BCM GPIO | Physical Pin | LED | Meaning |
|---|---|---|---|---|
| Green | GPIO 17 | Pin 11 | Green LED | Online (>10 GPM) |
| Amber | GPIO 22 | Pin 15 | Amber LED | Rinse (≤10 GPM) |
| Red | GPIO 27 | Pin 13 | Red LED | No flow (0 GPM) |

Each LED driven via 2N2222 transistor → 5m cable → 150Ω → LED → return.

### Quick GPIO Test

Run the cell below to generate a test script you can run on the Pi to verify wiring.

In [ ]:
# ── Generate GPIO + Camera test scripts for the Pi ────────────────
# These are written to files you can transfer and run on the Pi.

gpio_test_script = '''#!/usr/bin/env python3
"""Test script: Cycle through red, amber, green LEDs to verify wiring."""
import time
import RPi.GPIO as GPIO

GREEN_PIN = 17
AMBER_PIN = 22
RED_PIN = 27

GPIO.setmode(GPIO.BCM)
GPIO.setwarnings(False)
GPIO.setup(GREEN_PIN, GPIO.OUT)
GPIO.setup(AMBER_PIN, GPIO.OUT)
GPIO.setup(RED_PIN, GPIO.OUT)

def all_off():
    GPIO.output(GREEN_PIN, GPIO.LOW)
    GPIO.output(AMBER_PIN, GPIO.LOW)
    GPIO.output(RED_PIN, GPIO.LOW)

print("Cycling traffic lights... Press Ctrl+C to stop.")
try:
    for i in range(5):
        all_off()
        GPIO.output(RED_PIN, GPIO.HIGH)
        print(f"  [{i+1}] RED on    (no flow)")
        time.sleep(1.5)

        all_off()
        GPIO.output(AMBER_PIN, GPIO.HIGH)
        print(f"  [{i+1}] AMBER on  (rinse)")
        time.sleep(1.5)

        all_off()
        GPIO.output(GREEN_PIN, GPIO.HIGH)
        print(f"  [{i+1}] GREEN on  (online)")
        time.sleep(1.5)

    print("Test complete! All 3 LEDs working.")
except KeyboardInterrupt:
    print("\\nStopped.")
finally:
    all_off()
    GPIO.cleanup()
'''

camera_test_script = '''#!/usr/bin/env python3
"""Test script: Capture a single image from Pi Camera 2."""
import time
from picamera2 import Picamera2

cam = Picamera2()
config = cam.create_preview_configuration(main={"size": (640, 480), "format": "RGB888"})
cam.configure(config)
cam.start()
time.sleep(2)
cam.capture_file("camera_test.jpg")
cam.stop()
print("Saved camera_test.jpg — check that the rotameter is visible and in frame.")
'''

with open("test_gpio.py", "w") as f:
    f.write(gpio_test_script)
with open("test_camera.py", "w") as f:
    f.write(camera_test_script)

print("Generated test scripts for the Raspberry Pi:")
print("  test_gpio.py   — cycles red/amber/green LEDs to verify wiring")
print("  test_camera.py — captures one image to verify camera setup")
print()
print("Transfer to Pi and run:")
print("  scp test_gpio.py test_camera.py pi@<PI_IP>:~/lupw-project/")
print("  ssh pi@<PI_IP>")
print("  cd ~/lupw-project")
print("  python3 test_gpio.py")
print("  python3 test_camera.py")

## Section 11: Raspberry Pi Inference Script (TFLite Version)

The standalone inference script below runs on the Pi using TFLite (lighter than full TensorFlow). It includes:
- Camera capture loop
- TFLite model inference
- Dual-head zero-flow decision logic
- **3-state traffic light control:** RED (no flow) / AMBER (rinse ≤10 GPM) / GREEN (online >10 GPM)
- GPIO control for 3 LEDs (GPIO 17, 22, 27)
- **Debounce mechanism** — requires 3 consecutive zero-flow predictions before switching to red (prevents flickering)

In [ ]:
# ── Generate the TFLite inference script for the Pi ──────────────
inference_tflite_script = '''#!/usr/bin/env python3
"""
LUPW Flow Traffic Light System — Raspberry Pi TFLite Inference
Reads rotameter flow from Pi Camera 2, controls 3-state traffic light LEDs.

Traffic light states:
    RED   = No flow (0 GPM)
    AMBER = Rinse mode (<=10 GPM)
    GREEN = Online (>10 GPM)

Usage:
    python3 inference_tflite.py --model rotameter_model_fp16.tflite --max-flow 100

Hardware:
    Green LED: GPIO 17 (Pin 11)
    Amber LED: GPIO 22 (Pin 15)
    Red LED:   GPIO 27 (Pin 13)
"""

import argparse
import time
import csv
from datetime import datetime

import numpy as np
import cv2
from picamera2 import Picamera2
import RPi.GPIO as GPIO

# Try tflite_runtime first (lighter), fall back to full TF
try:
    from tflite_runtime.interpreter import Interpreter
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter

# ── Configuration ──────────────────────────────────────────────────
GREEN_PIN = 17
AMBER_PIN = 22
RED_PIN = 27
IMAGE_SIZE = (224, 224)
FLOW_THRESHOLD = 0.05   # Normalized: below 5% = near zero
RINSE_GPM = 10.0         # At or below this GPM = rinse (amber)
DEBOUNCE_COUNT = 3       # Consecutive zero readings before switching to red


def setup_gpio():
    GPIO.setmode(GPIO.BCM)
    GPIO.setwarnings(False)
    GPIO.setup(GREEN_PIN, GPIO.OUT)
    GPIO.setup(AMBER_PIN, GPIO.OUT)
    GPIO.setup(RED_PIN, GPIO.OUT)
    set_light("RED")


def set_light(state):
    """Set traffic light: 'RED', 'AMBER', or 'GREEN'."""
    GPIO.output(GREEN_PIN, GPIO.HIGH if state == "GREEN" else GPIO.LOW)
    GPIO.output(AMBER_PIN, GPIO.HIGH if state == "AMBER" else GPIO.LOW)
    GPIO.output(RED_PIN, GPIO.HIGH if state == "RED" else GPIO.LOW)


def is_zero_flow(flow_norm, zero_prob):
    if flow_norm < FLOW_THRESHOLD and zero_prob > 0.5:
        return True
    if zero_prob > 0.8:
        return True
    return False


def get_traffic_state(flow_gpm, zero_prob, flow_norm):
    """Determine traffic light state: RED / AMBER / GREEN."""
    if is_zero_flow(flow_norm, zero_prob):
        return "RED"
    elif flow_gpm <= RINSE_GPM:
        return "AMBER"
    else:
        return "GREEN"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default="rotameter_model_fp16.tflite")
    parser.add_argument("--max-flow", type=float, default=100.0)
    parser.add_argument("--rinse-gpm", type=float, default=RINSE_GPM)
    parser.add_argument("--interval", type=float, default=0.5)
    parser.add_argument("--log", default="flow_log.csv", help="CSV log file")
    args = parser.parse_args()

    global RINSE_GPM
    RINSE_GPM = args.rinse_gpm

    # Load TFLite model
    interpreter = Interpreter(model_path=args.model)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Setup hardware
    setup_gpio()
    cam = Picamera2()
    config = cam.create_preview_configuration(main={"size": (640, 480), "format": "RGB888"})
    cam.configure(config)
    cam.start()
    time.sleep(2)

    print("=" * 60)
    print("  LUPW Flow Traffic Light System (TFLite)")
    print("=" * 60)
    print(f"  Model:     {args.model}")
    print(f"  Max flow:  {args.max_flow} GPM")
    print(f"  Rinse thr: {args.rinse_gpm} GPM")
    print(f"  RED=0 GPM | AMBER=<=10 GPM | GREEN=>10 GPM")
    print("  Monitoring... Ctrl+C to stop.\\n")

    zero_counter = 0
    current_state = None

    # CSV logging
    log_file = open(args.log, "w", newline="")
    log_writer = csv.writer(log_file)
    log_writer.writerow(["timestamp", "flow_gpm", "zero_probability", "traffic_state"])

    try:
        while True:
            frame = cam.capture_array()
            # Preprocess
            img = cv2.resize(frame, IMAGE_SIZE)
            img = img.astype(np.float32) / 255.0
            img = np.expand_dims(img, axis=0)

            # Inference
            interpreter.set_tensor(input_details[0]["index"], img)
            interpreter.invoke()
            flow_norm = interpreter.get_tensor(output_details[0]["index"])[0][0]
            zero_prob = interpreter.get_tensor(output_details[1]["index"])[0][0]

            flow_gpm = float(np.clip(flow_norm, 0, 1) * args.max_flow)
            state = get_traffic_state(flow_gpm, zero_prob, flow_norm)

            # Debounce: require N consecutive zero readings before RED
            if state == "RED":
                zero_counter += 1
            else:
                zero_counter = 0

            if zero_counter < DEBOUNCE_COUNT and current_state == "RED":
                # Not enough consecutive zeros yet, keep previous non-red state
                pass
            elif state == "RED" and zero_counter < DEBOUNCE_COUNT:
                state = "AMBER"  # Show amber while debouncing toward red

            if state != current_state:
                set_light(state)
                current_state = state

            # Log
            now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log_writer.writerow([now, f"{flow_gpm:.2f}", f"{zero_prob:.3f}", state])

            STATE_LABELS = {"RED": "NO FLOW", "AMBER": "RINSE", "GREEN": "ONLINE"}
            print(f"\\r  Flow: {flow_gpm:6.2f}/{args.max_flow:.0f} GPM | "
                  f"Zero: {zero_prob:.2f} | {state}: {STATE_LABELS[state]}    ",
                  end="", flush=True)

            time.sleep(args.interval)

    except KeyboardInterrupt:
        print("\\n\\n  Shutting down...")
    finally:
        set_light("RED")
        GPIO.cleanup()
        cam.stop()
        log_file.close()
        print(f"  Log saved to {args.log}")
        print("  Goodbye!")


if __name__ == "__main__":
    main()
'''

with open("inference_tflite.py", "w") as f:
    f.write(inference_tflite_script)

print("Generated: inference_tflite.py")
print("Transfer to Pi: scp inference_tflite.py rotameter_model_fp16.tflite pi@<PI_IP>:~/lupw-project/")
print("Run on Pi: python3 inference_tflite.py --model rotameter_model_fp16.tflite --max-flow 100")

## Section 12: End-to-End Demo & Simulation

This cell demonstrates the full pipeline using test images (or simulated readings if not running on a Pi). It shows:
1. Flow prediction over time
2. Traffic light state changes
3. Real-time flow chart with zero-flow threshold line
4. CSV logging of all readings with timestamps

In [ ]:
# ── End-to-End Simulation ─────────────────────────────────────────
# Simulate a sequence of flow readings using test images (or synthetic data)
# This runs on the training machine — no Pi/GPIO needed

from datetime import datetime

# Use test set predictions if available, otherwise generate synthetic data
if len(all_flow_pred) > 20:
    sim_flows_norm = all_flow_pred[:50]
    sim_zero_probs = all_zero_pred[:50]
    sim_source = "Test set predictions"
else:
    # Generate synthetic flow scenario: online → rinse → zero → recover
    np.random.seed(42)
    sim_flows_norm = np.concatenate([
        np.random.uniform(0.4, 0.9, 10),     # Online flow (>10 GPM)
        np.linspace(0.4, 0.08, 8),            # Dropping toward rinse
        np.random.uniform(0.05, 0.10, 7),     # Rinse range (~5-10 GPM)
        np.linspace(0.08, 0.0, 5),            # Dropping to zero
        np.zeros(8) + np.random.normal(0, 0.01, 8),  # Zero flow
        np.linspace(0.0, 0.08, 5),            # Recovering to rinse
        np.linspace(0.08, 0.5, 5),            # Recovering to online
        np.random.uniform(0.4, 0.8, 7),       # Back online
    ])
    sim_flows_norm = np.clip(sim_flows_norm, 0, 1)
    sim_zero_probs = np.where(sim_flows_norm < 0.05, 0.9, 0.1) + np.random.normal(0, 0.05, len(sim_flows_norm))
    sim_zero_probs = np.clip(sim_zero_probs, 0, 1)
    sim_source = "Synthetic simulation"

# Simulate the 3-state traffic light logic with debounce
DEBOUNCE = 3
sim_flow_gpm = sim_flows_norm * FLOW_MAX
sim_lights = []
zero_counter = 0

for flow_n, zero_p in zip(sim_flows_norm, sim_zero_probs):
    flow_gpm = flow_n * FLOW_MAX
    state = get_traffic_state(flow_gpm, zero_p, flow_n)

    if state == "RED":
        zero_counter += 1
    else:
        zero_counter = 0

    if state == "RED" and zero_counter < DEBOUNCE:
        state = "AMBER"  # Debouncing toward red

    sim_lights.append(state)

# ── Plot: Flow over time with 3-state traffic light ──────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle(f"End-to-End Demo — 3-State Traffic Light ({sim_source})", fontsize=14)

timesteps = range(len(sim_flow_gpm))

# Flow chart with threshold lines
ax1.plot(timesteps, sim_flow_gpm, "b-o", markersize=4, label="Predicted Flow (GPM)")
ax1.axhline(y=RINSE_GPM, color="orange", linestyle="--", linewidth=2, label=f"Rinse threshold ({RINSE_GPM} GPM)")
ax1.axhline(y=FLOW_THRESHOLD * FLOW_MAX, color="red", linestyle="--", linewidth=1.5, label="Zero-flow threshold")
ax1.fill_between(timesteps, 0, FLOW_THRESHOLD * FLOW_MAX, alpha=0.08, color="red")
ax1.fill_between(timesteps, FLOW_THRESHOLD * FLOW_MAX, RINSE_GPM, alpha=0.06, color="orange")
ax1.set_ylabel(f"Flow (GPM, max={FLOW_MAX})")
ax1.set_title("Flow Reading Over Time")
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

# Traffic light state bars
color_map = {"RED": "red", "AMBER": "#FFB000", "GREEN": "green"}
colors = [color_map[l] for l in sim_lights]
ax2.bar(timesteps, [1] * len(sim_lights), color=colors, edgecolor="none")
ax2.set_ylabel("Traffic Light")
ax2.set_xlabel("Reading #")
ax2.set_yticks([])
ax2.set_title("Traffic Light: Red=No Flow | Amber=Rinse (≤10 GPM) | Green=Online (>10 GPM)")

plt.tight_layout()
plt.savefig("demo_simulation.png", dpi=150)
plt.show()

# ── Save simulation log to CSV ───────────────────────────────────
log_df = pd.DataFrame({
    "reading": list(timesteps),
    "flow_gpm": [f"{f:.2f}" for f in sim_flow_gpm],
    "zero_probability": [f"{z:.3f}" for z in sim_zero_probs],
    "traffic_state": sim_lights,
    "timestamp": [datetime.now().strftime("%Y-%m-%d %H:%M:%S") for _ in timesteps],
})

log_df.to_csv("demo_flow_log.csv", index=False)
print(f"Simulation log saved to demo_flow_log.csv ({len(log_df)} readings)")
print(f"\nSummary:")
print(f"  Total readings: {len(sim_lights)}")
print(f"  🟢 GREEN (online):  {sim_lights.count('GREEN')}")
print(f"  🟡 AMBER (rinse):   {sim_lights.count('AMBER')}")
print(f"  🔴 RED (no flow):   {sim_lights.count('RED')}")

## Next Steps

1. **Collect your dataset** — photograph the rotameter at various flow levels as described in Section 2
2. **Run this notebook** — train the model on your dataset
3. **Transfer to Pi** — copy `rotameter_model_fp16.tflite` to the Raspberry Pi
4. **Wire the hardware** — follow [setup_guide.md](setup_guide.md) for 3-LED wiring (green=GPIO 17, amber=GPIO 22, red=GPIO 27)
5. **Test hardware** — run `test_gpio.py` (cycles all 3 LEDs) and `test_camera.py` on the Pi
6. **Run inference** — `python3 inference_tflite.py --model rotameter_model_fp16.tflite --max-flow <YOUR_MAX>`
7. **Tune thresholds** — adjust `RINSE_GPM` (default 10) and `DEBOUNCE_COUNT` for your specific setup

### Traffic Light States
| State | Condition | LED |
|---|---|---|
| RED | No flow (0 GPM) | Red ON |
| AMBER | Rinse (>0 and ≤10 GPM) | Amber ON |
| GREEN | Online (>10 GPM) | Green ON |

### Key Parameters to Customize
| Parameter | Where | Description |
|---|---|---|
| `FLOW_MAX` | Training notebook | Your rotameter's maximum scale reading |
| `RINSE_GPM` | inference_tflite.py / `--rinse-gpm` | GPM threshold for rinse→online (default: 10) |
| `DEBOUNCE_COUNT` | inference_tflite.py | Consecutive zero readings before red (default: 3) |
| `--interval` | CLI flag | Seconds between readings (default: 0.5s) |